# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ishwarsolanki-004/ML-internship-with-flyrank-ai/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Health Score feature importance

The paper defines a Health Score using four hand-built components: impressions, average position, CTR, and scroll depth. Later, the Random Forest feature-importance analysis shows these same four components as the dominant features.

**Methodology question:** Since the Health Score is constructed directly from impressions, position, CTR, and scroll depth, could this feature-importance result be interpreted as discovering what makes content good, or is it mainly showing that the model can reconstruct the recipe used to create the score?

This is a useful distinction because the result describes how the model reproduces an existing score; it does not by itself establish why content performs well or what causes better performance.

### Finding 2 — 71% accuracy on an 80/20 holdout

The paper reports that the model classified 71% of the held-out content correctly using an 80/20 train-test split. The paper does not provide enough detail about how records were grouped across the split, whether brands appeared in both training and testing, the starting base rate, or whether the result was repeated across multiple splits.

**Methodology question:** Could the evaluation clarify whether the test set contains pages from brands also seen during training, and how much the 71% result improves over a naive baseline?

This would help determine whether the result mainly reflects performance on familiar brands or whether it supports a stronger claim about generalization to unseen brands.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

I first compare a row-level random split with the client-grouped split used in Week 5.

The random split is useful as a reference because rows from the same client can appear in both training and test sets. The grouped split keeps complete clients on one side of the split and therefore asks the stricter question: does the model rank declining content for clients it did not see during training?

I use the same target, feature set, model family, random seed, and Precision@50 metric for both comparisons.

The grouped result is treated as the more honest estimate for generalization to unseen clients. A gap between the random and grouped results is itself a finding rather than something to hide.


In [1]:
# ==========================================
# SECTION 2 — SETUP
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

DATA_PATH = "../../data/processed/refresh_feature_vector.csv"

df = pd.read_csv(DATA_PATH)

TARGET = "is_declining_label"

# Columns excluded from model features
EXCLUDE_COLUMNS = [
    "client_id",
    "content_id",
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in EXCLUDE_COLUMNS
]

# Features removed during the Week-5 leakage check
LEAKAGE_FEATURES = [
    "trend_pct",
    "trend_direction",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

CLEAN_FEATURE_COLUMNS = [
    col for col in FEATURE_COLUMNS
    if col not in LEAKAGE_FEATURES
]

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Original features:", len(FEATURE_COLUMNS))
print("Clean features:", len(CLEAN_FEATURE_COLUMNS))
print("Overall base rate:", round(df[TARGET].mean(), 4))

Rows: 30000
Clients: 32
Original features: 47
Clean features: 44
Overall base rate: 0.5421


In [2]:
# ==========================================
# MODEL BUILDER
# ==========================================

def build_logistic_model(X_train):
    
    numeric_features = X_train.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        exclude=["number", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ])

    return model

In [3]:
# ==========================================
# PRECISION@K
# ==========================================

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(scores))

    order = np.argsort(scores)[::-1]
    top_k_indices = order[:k]

    return y_true[top_k_indices].mean()

In [4]:
# ==========================================
# BEFORE — RANDOM ROW SPLIT
# ==========================================

X = df[CLEAN_FEATURE_COLUMNS].copy()
y = df[TARGET].astype(int)

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

random_model = build_logistic_model(X_train_random)

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_precision_20 = precision_at_k(
    y_test_random.values,
    random_scores,
    20
)

random_precision_50 = precision_at_k(
    y_test_random.values,
    random_scores,
    50
)

print("Random split train rows:", len(X_train_random))
print("Random split test rows:", len(X_test_random))
print("Random test base rate:", round(y_test_random.mean(), 4))

print(
    "Random split Precision@20:",
    round(random_precision_20, 4)
)

print(
    "Random split Precision@50:",
    round(random_precision_50, 4)
)

Random split train rows: 24000
Random split test rows: 6000
Random test base rate: 0.542
Random split Precision@20: 0.95
Random split Precision@50: 0.98


In [5]:
# ==========================================
# AFTER — CLIENT-GROUPED SPLIT
# ==========================================

clients = df["client_id"].drop_duplicates()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_df = df[
    df["client_id"].isin(train_clients)
].copy()

test_df = df[
    df["client_id"].isin(test_clients)
].copy()

X_train_grouped = train_df[CLEAN_FEATURE_COLUMNS].copy()
y_train_grouped = train_df[TARGET].astype(int)

X_test_grouped = test_df[CLEAN_FEATURE_COLUMNS].copy()
y_test_grouped = test_df[TARGET].astype(int)

grouped_model = build_logistic_model(X_train_grouped)

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_precision_20 = precision_at_k(
    y_test_grouped.values,
    grouped_scores,
    20
)

grouped_precision_50 = precision_at_k(
    y_test_grouped.values,
    grouped_scores,
    50
)

client_overlap = (
    set(train_clients) &
    set(test_clients)
)

print("Grouped train rows:", len(train_df))
print("Grouped test rows:", len(test_df))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))
print("Grouped test base rate:", round(y_test_grouped.mean(), 4))

print(
    "Client-grouped Precision@20:",
    round(grouped_precision_20, 4)
)

print(
    "Client-grouped Precision@50:",
    round(grouped_precision_50, 4)
)

Grouped train rows: 26581
Grouped test rows: 3419
Train clients: 25
Test clients: 7
Client overlap: 0
Grouped test base rate: 0.5238
Client-grouped Precision@20: 0.65
Client-grouped Precision@50: 0.7


In [6]:
# ==========================================
# BEFORE / AFTER COMPARISON
# ==========================================

comparison = pd.DataFrame({
    "validation": [
        "Random row split",
        "Client-grouped split"
    ],
    "test_rows": [
        len(X_test_random),
        len(X_test_grouped)
    ],
    "test_base_rate": [
        y_test_random.mean(),
        y_test_grouped.mean()
    ],
    "precision_at_20": [
        random_precision_20,
        grouped_precision_20
    ],
    "precision_at_50": [
        random_precision_50,
        grouped_precision_50
    ]
})

display(comparison.round(4))

,validation,test_rows,test_base_rate,precision_at_20,precision_at_50
0,Random row split,6000,0.5420,0.95,0.98
1,Client-grouped split,3419,0.5238,0.65,0.70


In [7]:
# ==========================================
# SPLIT GAP
# ==========================================

precision_50_gap = (
    random_precision_50 - grouped_precision_50
)

print(
    "Precision@50 gap (random - grouped):",
    round(precision_50_gap, 4)
)

Precision@50 gap (random - grouped): 0.28


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audited the final feature set for three main leakage risks: label-derived features, future or overlapping-window information, and decision-derived features.

The label is `is_declining_label`, which is derived from `trend_direction`. Therefore `trend_direction` and `trend_pct` are excluded from the model features.

The identifiers `client_id` and `content_id` are used for grouping, joining, and inspection only; they are not model features.

During the Week-5 model check, `impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d` were removed from the model. The original model produced perfect Precision@20 and Precision@50, while `impressions_last_30d` had a very large coefficient. After removing these label-aligned variables, the client-grouped result decreased to Precision@20 = 0.65 and Precision@50 = 0.70.

This change is treated as an audit finding rather than something to hide. It shows that the original metric was sensitive to features closely aligned with the label construction.

The remaining features are still interpreted cautiously. In particular, the starter label is a current-window proxy, so this evaluation should be treated as directional decision-support rather than proof of future performance or causation.

In [8]:
# ==========================================
# LEAKAGE AUDIT — FEATURE CHECK
# ==========================================

print("Label:", TARGET)

print("\nFeatures containing 'trend':")
print([
    col for col in CLEAN_FEATURE_COLUMNS
    if "trend" in col.lower()
])

print("\nFeatures containing 'label':")
print([
    col for col in CLEAN_FEATURE_COLUMNS
    if "label" in col.lower()
])

print("\nIdentifiers used only for grouping/inspection:")
print(["client_id", "content_id"])

print("\nFeatures removed during leakage audit:")
print(LEAKAGE_FEATURES)

print("\nRemaining 30d features:")
print([
    col for col in CLEAN_FEATURE_COLUMNS
    if "30d" in col.lower()
])

Label: is_declining_label

Features containing 'trend':
[]

Features containing 'label':
[]

Identifiers used only for grouping/inspection:
['client_id', 'content_id']

Features removed during leakage audit:
['trend_pct', 'trend_direction', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']

Remaining 30d features:
['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


In [9]:
# ==========================================
# CONFIRM REMOVED FEATURES ARE NOT IN MODEL
# ==========================================

removed_still_present = [
    col for col in LEAKAGE_FEATURES
    if col in CLEAN_FEATURE_COLUMNS
]

print("Removed leakage-risk features still present:")
print(removed_still_present)

print(
    "\nLeakage-risk feature check:",
    "PASS" if len(removed_still_present) == 0 else "REVIEW REQUIRED"
)

Removed leakage-risk features still present:
[]

Leakage-risk feature check: PASS


### Leakage audit conclusion

The final feature-set check found no remaining trend, label, or explicitly removed leakage-risk features. The audit also confirmed that client_id and content_id are not used as model features.

The identified last-30-day variables were removed because they were closely aligned with the label construction. After this change, the model's measured performance decreased substantially under the stricter client-grouped split.

The remaining `*_prev_30d` variables represent the previous 30-day window and are retained as candidate historical features. However, because the starter label is a current-window proxy, the model should still be interpreted cautiously. The leakage audit passes for the explicitly identified risks, but this does not establish causation or guarantee future performance.

Overall, the validated model is treated as directional decision-support for prioritizing content review rather than as a guarantee of which pages will decline.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

The model is better than the existing rule and can help any client identify pages that will decline next month.

### Rewritten claim

Observed and measured on the evaluated data, the cleaned Logistic Regression model provides a directional ranking signal for declining content.

Under the client-grouped validation, the model achieved a measured Precision@50 of 0.70, compared with 0.98 under the random row split. The 0.28 gap shows that the measured result is sensitive to validation design.

The client-grouped result is therefore the more conservative measurement for generalization to unseen clients. The model can be used as decision-support to help prioritize pages for review, but the result does not establish causation, guarantee future decline, or show that acting on the recommendations will prevent decline.

In [10]:
# ==========================================
# CLAIM SUPPORTING NUMBERS
# ==========================================

print("Random-split Precision@50:",
      round(random_precision_50, 4))

print("Client-grouped Precision@50:",
      round(grouped_precision_50, 4))

print("Precision@50 gap:",
      round(precision_50_gap, 4))

print("Client-grouped Precision@20:",
      round(grouped_precision_20, 4))

print("Random-split Precision@20:",
      round(random_precision_20, 4))

Random-split Precision@50: 0.98
Client-grouped Precision@50: 0.7
Precision@50 gap: 0.28
Client-grouped Precision@20: 0.65
Random-split Precision@20: 0.95


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.